# Validation v3 · the production of one cube

The production that every v3 method reads (the design `experiments/validation_v3_cantabrian_2026-10-29.md`, its
amendment of 2026-11-03): Ferrol's production notebook (`tide_boundary_comparison_ferrol.ipynb`), sections 0-8,
on the cube of one truth. `NB_V3` names it (`experiments/v3/sites.py`): `urdaibai_2017`, `urdaibai_2024`,
`foz_2016` or `eo_2020`. It writes the production extraction (`marea_extract.npz`: the intertidal pixels, their
wet/dry record and their distance from the mouth), MAREA's product (`marea_<tag>/`) and the no-delay and MAREA
inversions (`comparison_<tag>/products.npz`), where notebooks 01 and 05 read them. Built by
`notebooks/validation/_build_v3_production.py`; the code cells are the template's, changed only where noted.

## Parameters

From `experiments/v3/sites.py`: the cube, the period, the boundary (EOT20 + the nearest IOC gauge, none held out),
the sea's edge and the record floor of the lag fit. Changed from the template: everything comes from the truth's
entry, and the overpass times are the downloader's file (`products_<site>_10m/overpass_times.json`).

In [ ]:
import os, sys

sys.path.insert(0, os.getcwd())
from experiments.v3.sites import NEW, V3

KEY = os.environ.get("NB_V3", "foz_2016")          # the truth whose cube this run produces
assert KEY in NEW, f"{KEY!r} is not a new v3 cube; one of {NEW}"
C = V3[KEY]

# ── Which area, and over what period (experiments/v3/sites.py) ───────────
SITE         = C["site"]
TIME_EXTENT  = C["period"]
RESOLUTION_M = 10
CACHE        = C["cube"]
OUT_DIR      = C["out"]
OVERPASS_CACHE = C["overpass"]                     # the downloader's overpass times of every cube date
MIN_SCENES   = C["min_scenes"]

# ── The boundary: two switches ──────────────────────────────────────────
TIDE_MODEL   = C["tide_model"]
N_GAUGES     = C["n_gauges"]
JUDGE_GAUGES = C["judges"]
MOUTH_SIDE   = C["mouth_side"]

# ── Derived ─────────────────────────────────────────────────────────────
BOUNDARY_TAG = f"{(TIDE_MODEL or 'nomodel').lower()}_g{N_GAUGES or 0}"
print(f"v3 truth {KEY}: {SITE} · {TIME_EXTENT[0]} → {TIME_EXTENT[1]} @ {RESOLUTION_M} m")
print(f"  boundary  model={TIDE_MODEL}  gauges={N_GAUGES} (expected {C['expected_gauge']})  "
      f"held out={JUDGE_GAUGES}  → tag {BOUNDARY_TAG}")
print(f"  cache     {CACHE}")
print(f"  outputs   {OUT_DIR}")

## 0 · Setup

No Copernicus connection is needed: the cube is cached.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyintertidal as pit
from pyintertidal import SentinelCube, explain, viz, marea
from pyintertidal import tide_estimators as te
from pyintertidal.boundary import make_boundary, nearest_gauges
from pyintertidal.gauges import load_cached_ioc, despike

pit.net.use_system_certificates()
os.makedirs(OUT_DIR, exist_ok=True)
print("pyintertidal", pit.__version__)
log = explain.OpLog(f"{SITE} v3 {TIME_EXTENT[0][:4]}–{TIME_EXTENT[1][:4]}")

## 1 · Study area and its instruments

Changed from the template: the AOI is the download box of `experiments/download_cube_v3.py` (the registry polygons
leave part of these estuaries out). The gauges come from the IOC station list: the nearest one forms the boundary.

In [ ]:
from pyintertidal.aoi import AOI

aoi = AOI.from_bbox(**C["box"], name=f"{SITE} (v3 box)")
print(aoi)
lat_c, lon_c = aoi.centroid
print(f"  centroid {lat_c:.3f}, {lon_c:.3f}")

stations = {s["Code"]: s for s in json.load(
    open("data_v4/gauges/ioc_stations.json", encoding="utf-8"))}
near = nearest_gauges(lat_c, lon_c, 6)
members = [g for g in near if g[0] not in JUDGE_GAUGES][:(N_GAUGES or 0)]
member_codes = [g[0] for g in members]
print("\nIOC gauges around the site:")
for code_, name, km in near:
    role = ("BOUNDARY member" if code_ in member_codes else
            "JUDGE (held out)" if code_ in JUDGE_GAUGES else "unused")
    print(f"  {code_:5s} {name:28s} {km:5.1f} km   {role}")
viz.plot_aoi(aoi);

## 2 · The datacube

The merged cube of the truth's years (B03, B08, SCL at 10 m). Every later step streams it.

In [ ]:
WATER = "ndwi"
cube = SentinelCube(aoi, TIME_EXTENT, water=WATER, cache_path=CACHE,
                    resolution=RESOLUTION_M)
cube.ensure(None)                       # cached → no connection needed
transform, crs = cube.grid
H, W = cube.shape
px_km2 = abs(transform.a * transform.e) / 1e6
print(f"{len(cube.dates)} scenes · grid {H}×{W} @ {abs(transform.a):g} m · {crs}")
explain.dates_available(cube.dates, TIME_EXTENT)

## 3 · Water detection threshold

Otsu over the clearest scenes is printed; the adopted value is NDWI = 0.

In [ ]:
otsu, info = pit.water.calibrate_threshold(cube, n_scenes=12)
NDWI_THRESHOLD = 0.0
print(f"\nadopted NDWI threshold: {NDWI_THRESHOLD:+.3f}")

## 4 · Coastal stability, cloud screening, water frequency, intertidal zone

The production parameters.

In [ ]:
CLEAN_SCENE_MAX_BAD  = 0.05
STABLE_THRESHOLD     = 0.95
TRANSITION_BUFFER_PX = 10
CLOUD_THRESHOLD      = 0.10

reference_map, cloud_pct = pit.reference_and_clouds(
    cube, threshold=NDWI_THRESHOLD, bad_classes=pit.water.BAD_CLASSES,
    clean_scene_max_bad=CLEAN_SCENE_MAX_BAD, stable_threshold=STABLE_THRESHOLD,
    transition_buffer_px=TRANSITION_BUFFER_PX)
for code_, name in [(0, "transition"), (1, "stable water"), (2, "stable land")]:
    n = int((reference_map == code_).sum())
    print(f"  class {code_} ({name:13s}): {n:>8,} px  {n * px_km2:6.2f} km²")
usable_dates = pit.filter_dates(cloud_pct, cloud_threshold=CLOUD_THRESHOLD)
# MAREA's scene record: "transition" (the transition-zone rule above) or
# "all" (every scene of the period, clouds masked per pixel). The choice is
# the truth-free decision of experiments/v12_scene_rule_choice.py when it
# has been run (results/scene_rule_choice/decision.json).
import json as _json, os as _os
_DEC = "results/scene_rule_choice/decision.json"
SCENE_RULE = _json.load(open(_DEC))["choice"] if _os.path.exists(_DEC) else "transition"
print(f"MAREA scene record: {SCENE_RULE}")
print(f"{len(usable_dates)} usable dates")
log.record("reference map + cloud stats", kind="reduce",
           params={"stable_threshold": STABLE_THRESHOLD}, inputs=["cube"],
           outputs=["reference map", f"{len(usable_dates)} dates"])
viz.plot_reference_map(reference_map, transform, crs, aoi);

In [ ]:
MIN_OBS = max(8, round(0.05 * len(usable_dates)))
water_freq = pit.water_frequency(cube, dates=usable_dates,
                                 threshold=NDWI_THRESHOLD, min_obs=MIN_OBS)
transition = reference_map == 0
otsu_low, otsu_high = pit.multiotsu_window(water_freq, transition)
WF_LOW, WF_HIGH = 0.05, 0.95
polygon_mask = aoi.raster_mask(transform, crs, reference_map.shape)
intertidal = pit.intertidal_mask(water_freq, reference_map, WF_LOW, WF_HIGH,
                                 aoi_mask=polygon_mask)
area_km2 = intertidal.sum() * px_km2
print(f"multi-Otsu window [{otsu_low:.2f}, {otsu_high:.2f}]; adopted "
      f"[{WF_LOW}, {WF_HIGH}] → intertidal {int(intertidal.sum()):,} px = "
      f"{area_km2:.2f} km²")
log.record("intertidal mask", kind="mask", params={"wf": [WF_LOW, WF_HIGH]},
           outputs=["intertidal mask"], note=f"{area_km2:.2f} km²")
fig, axes = plt.subplots(1, 2, figsize=(19, 7))
viz.plot_water_frequency(water_freq, transform, crs, aoi, ax=axes[0])
viz.plot_intertidal(intertidal, transform, crs, aoi,
                    title=f"Intertidal zone · {area_km2:.2f} km²", ax=axes[1])
fig.tight_layout();

## 5 · The boundary

EOT20 and the nearest gauge, and their consensus, which is the boundary.

In [ ]:
boundary = make_boundary(aoi, tide_model=TIDE_MODEL, n_gauges=N_GAUGES,
                         period=TIME_EXTENT, exclude=JUDGE_GAUGES)
print(f"boundary: {boundary.name}")

# the alternatives, for the record (cached gauges → cheap)
options = {}
if TIDE_MODEL:
    options[f"model {TIDE_MODEL}"] = make_boundary(
        aoi, TIDE_MODEL, None, period=TIME_EXTENT, exclude=JUDGE_GAUGES,
        verbose=False)
if N_GAUGES:
    options[f"{N_GAUGES} nearest gauges"] = make_boundary(
        aoi, None, N_GAUGES, period=TIME_EXTENT, exclude=JUDGE_GAUGES,
        verbose=False)
if TIDE_MODEL and N_GAUGES:
    options["consensus"] = boundary

# a 5-minute grid of every option over the whole period: the record every
# metric below reads, shifted in time by interpolation when a clock applies
grid5 = pd.date_range(TIME_EXTENT[0], pd.Timestamp(TIME_EXTENT[1]) + pd.Timedelta(days=1),
                      freq="5min")
lev5 = {k: np.asarray(v.levels(grid5), float) for k, v in options.items()}
lev5["boundary"] = np.asarray(boundary.levels(grid5), float)
for k, v in lev5.items():
    print(f"  {k:22s} range [{np.nanmin(v):+.2f}, {np.nanmax(v):+.2f}] m, "
          f"finite {np.isfinite(v).mean():.1%}")

# three days around a spring tide
rng_day = pd.Series(lev5["boundary"], index=grid5).resample("1D").agg(
    lambda x: np.nanmax(x) - np.nanmin(x))
d0 = rng_day.idxmax() - pd.Timedelta(days=1)
win = (grid5 >= d0) & (grid5 < d0 + pd.Timedelta(days=3))
fig, ax = plt.subplots(figsize=(15, 4.2))
for k, v in lev5.items():
    if k == "boundary":
        continue
    ax.plot(grid5[win], v[win], lw=1.4, label=k)
ax.set_ylabel("level (m, demeaned / model MSL)")
ax.set_title(f"The boundary options over a spring tide ({d0.date()})",
             loc="left"); ax.grid(alpha=0.3); ax.legend();

## 6 · Tide at each scene, and is the site mappable?

In [ ]:
# the catalogue lookup, cached on disk: a Sentinel-2 STAC outage must not
# cost a 40-minute run (it did, twice)
assert os.path.exists(OVERPASS_CACHE), f"{OVERPASS_CACHE}: the downloader writes it"
if os.path.exists(OVERPASS_CACHE):
    overpass = {k: pd.Timestamp(v).to_pydatetime() for k, v in
                json.load(open(OVERPASS_CACHE)).items()}
    print(f"overpass times from cache: {len(overpass)} dates")
else:
    overpass = pit.overpass.get_overpass_times(aoi.bbox, TIME_EXTENT, verbose=False)
    json.dump({k: v.isoformat() for k, v in overpass.items()}, open(OVERPASS_CACHE, "w"))
    print(f"overpass times from the catalogue: {len(overpass)} dates (cached)")
record_pool = (usable_dates if SCENE_RULE == "transition" else
               [d for d in cube.dates if TIME_EXTENT[0] <= d <= TIME_EXTENT[1]])
dated = [d for d in record_pool if d in overpass]
t_over = pd.DatetimeIndex(pd.to_datetime([overpass[d] for d in dated])).tz_localize(None)
h_over = np.asarray(boundary.levels(t_over), float)
tide_heights = {d: float(h) for d, h in zip(dated, h_over) if np.isfinite(h)}
print(f"{len(tide_heights)} of {len(usable_dates)} usable dates have an overpass "
      f"time and a finite boundary level")

series_times = grid5[::12]                      # hourly
series_heights = lev5["boundary"][::12]
sampled = np.array(list(tide_heights.values()))
metrics = pit.coverage.coverage_metrics(sampled, series_heights)
verdict = pit.coverage.coverage_verdict(metrics)
pit.coverage.print_coverage_report(metrics, verdict,
                                   title=f"Tidal sampling · {aoi.name}")
sampled_times = pd.to_datetime([overpass[d] for d in sorted(tide_heights)]).tz_localize(None)
sampled = np.array([tide_heights[d] for d in sorted(tide_heights)])
viz.plot_tide_record(series_times, series_heights, sampled_times, sampled,
                     model_name=boundary.name, location=(lat_c, lon_c));
log.record("boundary levels", kind="apply",
           params={"tide_model": TIDE_MODEL, "n_gauges": N_GAUGES,
                   "held_out": JUDGE_GAUGES},
           outputs=[f"{len(tide_heights)} scene levels"])

## 7 · MAREA — the interior clock, measured from the imagery

The binary wet/dry record of every intertidal pixel is extracted once from
all cube scenes (it fixes the pixels and the distance axis). MAREA's record is
then the usable dates of section 3 (the transition-zone cloud screening of the
method) that have an overpass time and a finite boundary level; inside a kept
scene, cloudy pixels are masked by the scene classification. MAREA bands the
flat by geodesic distance from the mouth, fits one clock lag per band against
the boundary (monotone inland, with a scene-bootstrap interval), smooths it to
every pixel and re-inverts every elevation. The "no delay" product is the same
inversion on the same record with the lag set to zero.

In [ ]:
EXTRACT = f"{OUT_DIR}/marea_extract.npz"
if os.path.exists(EXTRACT):
    z_ = np.load(EXTRACT, allow_pickle=True)
    ex = {k: z_[k] for k in z_.files}
    ex["shape"] = tuple(int(v) for v in ex["shape"])
    ex["bbox"] = ex["bbox"].item()
    ex["crs_wkt"] = str(ex["crs_wkt"])
    print(f"loaded {EXTRACT}")
else:
    ex = marea.extract(cube.cache_path)
    seeds = pit.geometry.mouth_seeds(ex["sea"])
    s_m = pit.geometry.along_distance(ex["sea"] | ex["inter"], seeds, RESOLUTION_M)
    ex["s_km"] = (s_m.ravel()[ex["keep"]] / 1000.0).astype(np.float32)
    np.savez_compressed(EXTRACT, Y=ex["Y"], C=ex["C"], keep=ex["keep"],
                        dates=ex["dates"], shape=np.array(ex["shape"]),
                        sea=ex["sea"], inter=ex["inter"], s_km=ex["s_km"],
                        bbox=np.array(ex["bbox"], dtype=object),
                        crs_wkt=np.array(ex["crs_wkt"]))
    print(f"extracted and saved {EXTRACT}")
keep = ex["keep"]
assert ex["shape"] == (H, W), "extraction grid differs from the cube grid"
# distance from the mouth, measured from the sea on MOUTH_SIDE only, and
# written back so the background job shares it
s_old = ex.get("s_km")
seeds = pit.geometry.mouth_seeds(ex["sea"], side=MOUTH_SIDE)
s_m = pit.geometry.along_distance(ex["sea"] | ex["inter"], seeds, RESOLUTION_M)
ex["s_km"] = (s_m.ravel()[keep] / 1000.0).astype(np.float32)
s_km = ex["s_km"].astype(float)
# rewrite the (large) extraction only when the distance axis changed
if s_old is None or not np.array_equal(np.asarray(s_old, np.float32), ex["s_km"], equal_nan=True):
    np.savez_compressed(EXTRACT, Y=ex["Y"], C=ex["C"], keep=keep, dates=ex["dates"],
                        shape=np.array(ex["shape"]), sea=ex["sea"], inter=ex["inter"],
                        s_km=ex["s_km"], bbox=np.array(ex["bbox"], dtype=object),
                        crs_wkt=np.array(ex["crs_wkt"]))
    print(f"distance axis updated -> rewrote {EXTRACT}")
print(f"{len(keep):,} intertidal px in the record "
      f"(pipeline mask above: {int(intertidal.sum()):,}); "
      f"s from the {MOUTH_SIDE} mouth up to {np.nanmax(s_km):.1f} km "
      f"({np.isnan(s_km).mean():.0%} hydraulically isolated px, no s)")

# the record: the method's transition-zone scene rule (usable dates of cell 12)
# ∩ an overpass time ∩ a finite boundary level (cell 17's tide_heights)
rec = np.array([d in tide_heights for d in ex["dates"]])
ex_rec = {**ex, "Y": ex["Y"][rec], "C": ex["C"][rec], "dates": ex["dates"][rec]}
print(f"MAREA record: {int(rec.sum())} of {len(rec)} cube scenes "
      f"(transition-zone rule, overpass time, finite boundary level)")

t0 = time.time()
marea_res = marea.reconstruct(cube.cache_path, out_dir=f"{OUT_DIR}/marea_{BOUNDARY_TAG}",
                              name=aoi.name, pixel_m=RESOLUTION_M,
                              boundary=boundary, extraction=ex_rec,
                              mouth_side=MOUTH_SIDE, min_scenes=MIN_SCENES,
                              overpass_times=overpass)
assert marea_res is not None and marea_res["estado"] == "ok", "reconstruct refused the record"
assert marea_res["n_escenas"] == int(rec.sum()), "the overpass lookup dropped record scenes"
print(f"\nscenes in the MAREA record: {marea_res['n_escenas']}")
print(f"pixels inverted: {marea_res['n_px_cota']:,} / {marea_res['n_px']:,}")
print(f"interior tide adopted: {marea_res['con_operador']}  "
      f"(physics veto: {marea_res['veto_fisico']})  [{time.time() - t0:.0f} s]")
mp = np.load(f"{OUT_DIR}/marea_{BOUNDARY_TAG}/marea.npz")
band_of, centres = mp["band"].astype(int), np.asarray(marea_res["centros_km"])
tau_fit, tau_used = np.asarray(marea_res["tau_min"]), np.asarray(marea_res["tau_usado_min"])
tau_lo, tau_hi = np.asarray(marea_res["tau_lo_min"]), np.asarray(marea_res["tau_hi_min"])
tau_px = mp["tau_px_min"].astype(float)          # the lag every pixel received
BAND_KM = float(mp["band_km"][0])
print(f"{len(centres)} bands of {BAND_KM:.0f} km; lag per band (min): {np.round(tau_used, 0)}")
print(f"95 % bootstrap intervals ({marea_res['n_boot']} scene resamples): "
      + ", ".join(f"[{a:+.0f}, {b:+.0f}]" for a, b in zip(tau_lo, tau_hi)))

s_grid = np.linspace(0, float(np.nanmax(s_km)), 200)
tau_curve = te.smooth_lag_profile(centres, tau_used, s_grid, bandwidth_km=BAND_KM)
plt.figure(figsize=(7.5, 4.2))
plt.errorbar(centres, tau_used, yerr=[np.clip(tau_used - tau_lo, 0, None), np.clip(tau_hi - tau_used, 0, None)],
             fmt="o", color="#1a76b3", capsize=4, label="lag per band (95 % bootstrap interval)")
plt.plot(s_grid, tau_curve, "-", color="crimson", lw=2, label=f"lag applied per pixel (kernel {BAND_KM:.0f} km)")
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("distance from the mouth  s  (km)"); plt.ylabel("cotidal lag  (min)")
plt.title(f"MAREA clocks · boundary = {boundary.name}"); plt.legend(); plt.grid(alpha=0.3);
log.record("MAREA clocks", kind="fit", params={"n_bands": len(centres)},
           inputs=["wet/dry record", "boundary"],
           outputs=[f"tau per band: {np.round(tau_used, 0).tolist()} min"])

## 8 · Bathymetry without vs with the MAREA adjustment

**Same pixels, same scenes, same inversion.** The plain product reads the
boundary at the overpass time; the MAREA product reads it τ minutes earlier
in each band where τ was detected. The elevation grid (`lo`, `hi`) is shared,
so the two maps differ only through the clock.

In [ ]:
times_all = overpass                    # the catalogue lookup of section 6, reused
have = rec.copy()                       # the MAREA record of cell 19 (transition-zone rule)
t_real = pd.DatetimeIndex(pd.to_datetime([times_all[d] for d in ex["dates"][have]])).tz_localize(None)
Y = np.nan_to_num(ex["Y"][have], nan=0.0).astype(np.float32)
C = ex["C"][have].astype(np.float32)
h0 = np.asarray(boundary.levels(t_real), float)
okh = np.isfinite(h0)
Y, C, t_real, h0 = Y[okh], C[okh], t_real[okh], h0[okh]
lo, hi = float(h0.min()), float(h0.max())
assert len(t_real) == marea_res["n_escenas"] and np.allclose([lo, hi], marea_res["rango_marea_m"]), \
    "the inversion record differs from the lag fit's record"
print(f"record: {len(t_real)} scenes × {Y.shape[1]:,} px; level grid [{lo:+.2f}, {hi:+.2f}] m")

def to_raster(vals):
    r = np.full(H * W, np.nan, np.float32); r[keep] = vals
    return r.reshape(H, W)

t0 = time.time()
z_plain, sg_plain = marea.invert_series(Y, C, h0, lo, hi, min_bracket=marea.QUALITY_MIN_BRACKET)
print(f"plain inversion: {int(np.isfinite(z_plain).sum()):,} px  [{time.time() - t0:.0f} s]")

# every pixel is inverted with the level series read tau_px minutes
# earlier, tau_px being the smoothed band profile at the pixel's own s;
# pixels sharing a (whole-minute) lag share the shifted series
z_marea = np.full(len(keep), np.nan)
for tv in np.unique(tau_px):
    cols = np.flatnonzero(tau_px == tv)
    h_k = (np.asarray(boundary.levels(t_real - pd.Timedelta(minutes=float(tv))), float)
           if tv else h0)
    z_marea[cols], _ = marea.invert_series(Y[:, cols], C[:, cols], h_k, lo, hi, min_bracket=marea.QUALITY_MIN_BRACKET)
print(f"MAREA inversion: {int(np.isfinite(z_marea).sum()):,} px, "
      f"{len(np.unique(tau_px))} distinct lags  [{time.time() - t0:.0f} s]")
# no detection threshold any more: the fitted profile IS the applied one;
# the alias keeps the downstream cells of this notebook unchanged
z_marea_fit = z_marea.copy()
# the shipped MAREA product used the identical call → it must agree
z_ship = mp["z"].astype(float)
m = np.isfinite(z_ship) & np.isfinite(z_marea)
dd = np.abs(z_ship[m] - z_marea[m])
print(f"agreement with the shipped product: median |Δ| = {np.median(dd):.3f} m, "
      f"p95 {np.percentile(dd, 95):.3f} m, {np.mean(dd > 0.05):.1%} of px differ by > 5 cm")

dem_plain, dem_marea, dem_fit = to_raster(z_plain), to_raster(z_marea), to_raster(z_marea_fit)
fig, axes = plt.subplots(2, 2, figsize=(19, 13))
viz.plot_dem(dem_plain, transform, crs, aoi, title="Without adjustment (boundary at overpass time)", ax=axes[0, 0])
viz.plot_dem(dem_marea, transform, crs, aoi, title="With MAREA clocks (applied product)", ax=axes[0, 1])
viz.plot_map(dem_marea - dem_plain, transform, crs, aoi, cmap="RdBu_r", vmin=-0.3, vmax=0.3,
             robust=False, hillshade=False, title="applied MAREA − plain (m)", cbar_label="Δz (m)", ax=axes[1, 0])
viz.plot_map(to_raster(tau_px), transform, crs, aoi, cmap="magma", vmin=0, vmax=max(1.0, float(tau_px.max())),
             robust=False, hillshade=False, title="lag applied to each pixel (min)", cbar_label="τ (min)", ax=axes[1, 1])
fig.tight_layout();

In [ ]:
# where the clock acts: the difference by band of distance from the mouth
# (the fitted clocks, so the table says something even when the applied
# product keeps the boundary's clock everywhere)
d = z_marea_fit - z_plain
rows_ = []
for k in range(len(centres)):
    sel = (band_of == k) & np.isfinite(d)
    rows_.append({"band": k, "s_km": round(float(centres[k]), 2),
                  "tau_fitted_min": round(float(tau_fit[k]), 1),
                  "tau_applied_min": float(tau_used[k]), "n_px": int(sel.sum()),
                  "median_dz_m": float(np.median(d[sel])) if sel.any() else np.nan,
                  "p10_dz_m": float(np.percentile(d[sel], 10)) if sel.any() else np.nan,
                  "p90_dz_m": float(np.percentile(d[sel], 90)) if sel.any() else np.nan})
band_table = pd.DataFrame(rows_)
display(band_table.round(3))
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(band_table.s_km, band_table.median_dz_m,
            yerr=[band_table.median_dz_m - band_table.p10_dz_m, band_table.p90_dz_m - band_table.median_dz_m],
            fmt="o-", color="#1a76b3", capsize=3, label="median Δz (p10–p90)")
ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("s from the mouth (km)"); ax.set_ylabel("fitted clocks − plain (m)")
ax.set_title("What the fitted clock would do to the elevations, by band", loc="left"); ax.grid(alpha=0.3); ax.legend();

## 9 · The products and their provenance

What notebooks 01 and 05 read: `comparison_<tag>/products.npz` (the no-delay inversion `z_plain` gives the range
the truth is kept in), and `v3_production.json` (the inputs, the boundary, the record and the lags).

In [ ]:
import subprocess

prod_dir = f"{OUT_DIR}/comparison_{BOUNDARY_TAG}"
os.makedirs(prod_dir, exist_ok=True)
np.savez_compressed(f"{prod_dir}/products.npz",
                    keep=keep, shape=np.array([H, W]), s_km=s_km, band=band_of,
                    z_plain=z_plain, z_marea=z_marea, z_marea_fit=z_marea_fit,
                    record_dates=np.asarray(ex["dates"][have])[okh], level_range=np.array([lo, hi]))
band_table.to_csv(f"{prod_dir}/marea_effect_by_band.csv", index=False)
summary = pd.DataFrame([{"product": name, "px mapped": int(np.isfinite(z).sum()),
                         "median z (m)": float(np.nanmedian(z)),
                         "median |Δz| vs plain (m)": float(np.nanmedian(np.abs(z - z_plain)))}
                        for name, z in (("no delay", z_plain), ("MAREA", z_marea))])
display(summary.round(3))


def git(*args):
    try:
        return subprocess.run(["git", *args], capture_output=True, text=True, check=True).stdout.strip()
    except Exception:  # noqa: BLE001 - provenance only
        return None


record = {
    "key": KEY, "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in C.items()},
    "design": "experiments/validation_v3_cantabrian_2026-10-29.md (amendment of 2026-11-03)",
    "template": "tide_boundary_comparison_ferrol.ipynb (sections 0-8, built by notebooks/validation/_build_v3_production.py)",
    "git_commit": git("rev-parse", "HEAD"), "git_dirty": bool(git("status", "--porcelain", "--untracked-files=no")),
    "cube": {"path": CACHE, "bytes": os.path.getsize(CACHE), "scenes": len(cube.dates), "shape": [H, W]},
    "usable_dates": len(usable_dates), "scene_rule": SCENE_RULE, "ndwi_threshold": NDWI_THRESHOLD,
    "boundary": boundary.name, "boundary_members": member_codes,
    "marea": {"record_scenes": int(marea_res["n_escenas"]), "pixels": int(marea_res["n_px"]),
              "pixels_with_z": int(marea_res["n_px_cota"]), "band_centres_km": [float(c) for c in centres],
              "tau_fit_min": [float(t) for t in tau_fit], "tau_used_min": [float(t) for t in tau_used]},
    "extraction": {"path": EXTRACT, "keep_px": int(len(keep)), "dates": int(len(ex["dates"]))},
    "products": f"{prod_dir}/products.npz", "marea_dir": f"{OUT_DIR}/marea_{BOUNDARY_TAG}",
    "pyintertidal": pit.__version__,
}
json.dump(record, open(f"{OUT_DIR}/v3_production.json", "w", encoding="utf-8"), indent=1, default=str)
assert C["expected_gauge"] in member_codes, f"boundary gauge {member_codes}, expected {C['expected_gauge']}"
print(f"-> {prod_dir}/products.npz, {OUT_DIR}/marea_{BOUNDARY_TAG}/ and {OUT_DIR}/v3_production.json")